# Miniproyecto 1: Clasificación de tumores cerebrales en MRI con CNN

**Curso:** Técnicas de Deep Learning (MAIA-4213), Maestría en Inteligencia Artificial, Universidad de los Andes.

**Objetivo:** desarrollar un método basado en redes neuronales convolucionales que clasifique imágenes 2D extraídas de MRI en una de cuatro categorías: *glioma*, *meningioma*, *pituitary* y *healthy*.

**Dataset:** [Brain Tumor (MRI Scans), Kaggle](https://www.kaggle.com/datasets/rm1000/brain-tumor-mri-scans), 7023 imágenes, versionado en `data/raw/`.

## Estructura del notebook

| Sección | Contenido |
|---|---|
| 0 | Configuración y reproducibilidad |
| 1 | Inventario y exploración del dataset |
| 2 | Detección de duplicados y casi-duplicados |
| 3 | Partición train / validación / test por grupos |
| 4 | Preprocesamiento, aumento de datos y carga |
| 5 | Modelos: CNN propia y ResNet18 preentrenada |
| 6 | Funciones de entrenamiento y evaluación |
| 7 | Experimento 1: ablación de la CNN propia (profundidad x tamaño de filtro) |
| 8 | Experimento 2: transferencia de aprendizaje (ResNet18) |
| 9 | Experimento 3: robustez con varias semillas |
| 10 | Experimento 4: efecto de la fuga de datos (split aleatorio vs por grupos) |
| 11 | Resultados cuantitativos finales |
| 12 | Resultados cualitativos y Grad-CAM |
| 13 | Resumen de cifras para el informe |

## Cómo ejecutarlo

1. **Local con GPU NVIDIA:** clonar el repositorio, instalar PyTorch con CUDA ([pytorch.org](https://pytorch.org/get-started/locally/)) y `pip install -r requirements.txt`. Abrir este notebook desde la carpeta `notebooks/` y ejecutar todas las celdas.
2. **Google Colab:** Entorno de ejecución > Cambiar tipo > GPU. La celda 0.1 clona el repositorio con los datos automáticamente.

Todas las salidas (tablas, figuras y pesos) se guardan en `outputs/`. Si la ejecución se interrumpe, al volver a correr el notebook los modelos ya entrenados se cargan desde `outputs/models/` en lugar de reentrenarse.

## 0. Configuración y reproducibilidad

In [ ]:
# 0.1 Solo en Google Colab: clona el repositorio (incluye los datos en data/raw)
import sys, os, subprocess
from pathlib import Path

if "google.colab" in sys.modules and not Path("/content/brain-tumor-mri-cnn").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/panamiguell/brain-tumor-mri-cnn.git",
                    "/content/brain-tumor-mri-cnn"], check=True)
if "google.colab" in sys.modules:
    os.chdir("/content/brain-tumor-mri-cnn/notebooks")

In [ ]:
# 0.2 Librerías
import json, time, random, hashlib, itertools, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import v2 as T

from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             precision_recall_fscore_support, classification_report)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.precision", 4)
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)

In [ ]:
# 0.3 Rutas del proyecto (funciona desde notebooks/ o desde la raíz del repo)
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").exists(), "No se encontró data/raw. Ejecute el notebook dentro del repositorio."

# Modo rápido: solo para verificar que el código corre de principio a fin (subconjunto, 1 época).
# Para los resultados reales debe quedar en False.
QUICK = os.environ.get("MP1_QUICK", "0") == "1"

RAW    = ROOT / "data" / "raw"
SPLITS = ROOT / "data" / "splits"
CACHE  = ROOT / "data" / "cache"
OUT    = ROOT / "outputs" / ("_smoke_test" if QUICK else "")
FIG, TAB, MODELS = OUT / "figures", OUT / "tables", OUT / "models"
for d in [SPLITS, CACHE, FIG, TAB, MODELS]:
    d.mkdir(parents=True, exist_ok=True)

CLASSES = ["glioma", "meningioma", "pituitary", "healthy"]
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
TUMOR_IDX = [CLS2IDX[c] for c in ["glioma", "meningioma", "pituitary"]]
HEALTHY_IDX = CLS2IDX["healthy"]

CFG = dict(
    seed=42,
    img_size=224,            # resolución de entrada a las redes
    batch_size=32,
    n_folds_test=7,          # 1/7 de los grupos a test (~14 %)
    n_folds_val=6,           # 1/6 del resto a validación (~14 % del total)
    dup_corr_threshold=0.95, # correlación mínima para considerar dos imágenes copias del mismo corte
    epochs_cnn=25, lr_cnn=1e-3,
    epochs_tl=12,  lr_tl=1e-4,
    weight_decay=1e-4, patience=6, label_smoothing=0.0,
    ablation_blocks=[2, 3, 4], ablation_kernels=[3, 5],
    seeds=[42, 123, 2024],
    resume=True,             # reutiliza pesos ya entrenados en outputs/models
)
if QUICK:
    CFG.update(epochs_cnn=1, epochs_tl=1, ablation_blocks=[2, 3], ablation_kernels=[3],
               seeds=[42, 123], batch_size=16, resume=False)
    QUICK_N = dict(train=160, val=64, test=64)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Raíz:", ROOT)
print("Dispositivo:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "sin GPU")
print("Modo rápido:", QUICK)
if DEVICE.type != "cuda" and not QUICK:
    print("ADVERTENCIA: no se detectó GPU; el entrenamiento completo será muy lento.")

In [ ]:
# 0.4 Semillas para reproducibilidad
def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True  # más rápido; puede introducir variaciones mínimas entre ejecuciones

set_seed(CFG["seed"])

## 1. Inventario y exploración del dataset

Se recorre `data/raw/<clase>/` y para cada imagen se registra: tamaño, modo de color, intensidad media, hash MD5 (para detectar copias exactas) y un *difference hash* de 64 bits (para detectar candidatos a casi-duplicados).

In [ ]:
def dhash(img_gray: Image.Image) -> str:
    # Difference hash de 64 bits: compara píxeles vecinos en una miniatura de 9x8
    a = np.asarray(img_gray.resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return "".join("1" if v else "0" for v in (a[:, 1:] > a[:, :-1]).ravel())

INV_PATH = SPLITS / "inventario.csv"
if INV_PATH.exists():
    inv = pd.read_csv(INV_PATH, dtype={"dhash": str})
    print("Inventario cargado desde", INV_PATH.relative_to(ROOT))
else:
    rows = []
    for c in CLASSES:
        for p in tqdm(sorted((RAW / c).glob("*.jpg")), desc=c):
            b = p.read_bytes()
            with Image.open(p) as im:
                w, h, mode = im.size[0], im.size[1], im.mode
                g = im.convert("L")
                rows.append(dict(path=p.relative_to(ROOT).as_posix(), clase=c, label=CLS2IDX[c],
                                 w=w, h=h, mode=mode, mean=float(np.asarray(g).mean()),
                                 md5=hashlib.md5(b).hexdigest(), dhash=dhash(g)))
    inv = pd.DataFrame(rows)

print("Total de imágenes:", len(inv))
resumen = inv.groupby("clase").agg(imagenes=("path", "size"),
                                   ancho_mediana=("w", "median"), alto_mediana=("h", "median"),
                                   intensidad_media=("mean", "mean")).loc[CLASSES]
resumen["%"] = (100 * resumen.imagenes / len(inv)).round(1)
display(resumen)
print("Modos de color:", inv["mode"].value_counts().to_dict())
print("Tamaños distintos:", inv.groupby(["w", "h"]).ngroups)

In [ ]:
# Distribución de clases y de resoluciones por clase
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
resumen["imagenes"].plot.bar(ax=ax[0], color="#4C72B0")
ax[0].set_title("Imágenes por clase"); ax[0].set_ylabel("Número de imágenes"); ax[0].tick_params(axis="x", rotation=0)
for c in CLASSES:
    s = inv[inv.clase == c]
    ax[1].scatter(s.w, s.h, s=6, alpha=0.4, label=c)
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("Ancho (px)"); ax[1].set_ylabel("Alto (px)"); ax[1].set_title("Resolución original por clase"); ax[1].legend(markerscale=3)
plt.tight_layout(); plt.savefig(FIG / "eda_clases_resolucion.png", dpi=200); plt.show()

In [ ]:
# Ejemplos aleatorios por clase
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(12, 8.5))
for i, c in enumerate(CLASSES):
    for j, p in enumerate(inv[inv.clase == c].sample(6, random_state=CFG["seed"]).path):
        axes[i, j].imshow(Image.open(ROOT / p).convert("L"), cmap="gray"); axes[i, j].axis("off")
    axes[i, 0].set_title(c, loc="left", fontsize=11)
plt.tight_layout(); plt.savefig(FIG / "eda_ejemplos.png", dpi=150); plt.show()

## 2. Detección de duplicados y casi-duplicados

El dataset no incluye identificador de paciente. Si dos copias del mismo corte quedan una en entrenamiento y otra en test, el modelo es evaluado sobre imágenes que ya vio y las métricas se inflan (fuga de datos).

Se agrupan las imágenes en **grupos de copias** con dos criterios:
1. **Copias exactas:** mismo hash MD5.
2. **Casi-duplicados:** mismo *difference hash* y correlación de Pearson entre píxeles (miniaturas de 128x128) mayor o igual a `dup_corr_threshold`. Esto captura copias reescaladas o con contraste modificado.

Como los candidatos a casi-duplicado se buscan por coincidencia exacta del *difference hash*, el número de copias encontrado es un **límite inferior**.

In [ ]:
def thumb_z(path: str) -> np.ndarray:
    a = np.asarray(Image.open(ROOT / path).convert("L").resize((128, 128), Image.BILINEAR), dtype=np.float32).ravel()
    return (a - a.mean()) / (a.std() + 1e-6)

if "grupo" not in inv.columns:
    parent = list(range(len(inv)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra

    for _, g in inv.groupby("md5"):          # 1) copias exactas
        idx = list(g.index)
        for j in idx[1:]: union(idx[0], j)

    pares = []                                # 2) casi-duplicados
    cache_z = {}
    for _, g in tqdm(inv.groupby("dhash"), desc="pares candidatos"):
        if len(g) < 2: continue
        for a, b in itertools.combinations(g.itertuples(), 2):
            if a.md5 == b.md5: continue
            za = cache_z.setdefault(a.Index, thumb_z(a.path))
            zb = cache_z.setdefault(b.Index, thumb_z(b.path))
            r = float(np.dot(za, zb) / za.size)
            pares.append((a.clase, b.clase, r))
            if r >= CFG["dup_corr_threshold"]:
                union(a.Index, b.Index)
    inv["grupo"] = [find(i) for i in range(len(inv))]
    inv.to_csv(INV_PATH, index=False)

    P = pd.DataFrame(pares, columns=["c1", "c2", "corr"])
    P["misma_clase"] = P.c1 == P.c2
    print("Pares candidatos analizados:", len(P))
    print("Correlación máxima entre clases distintas:", round(P.loc[~P.misma_clase, "corr"].max(), 3)
          if (~P.misma_clase).any() else "sin pares")
    print("Pares sobre el umbral entre clases distintas:",
          int(((~P.misma_clase) & (P["corr"] >= CFG["dup_corr_threshold"])).sum()))

dup = inv.groupby("clase").agg(imagenes=("path", "size"), cortes_unicos=("grupo", "nunique")).loc[CLASSES]
dup["redundantes"] = dup.imagenes - dup.cortes_unicos
dup["%_redundante"] = (100 * dup.redundantes / dup.imagenes).round(1)
dup.to_csv(TAB / "duplicados_por_clase.csv")
display(dup)
print("Grupos con imágenes de más de una clase:", int((inv.groupby("grupo").clase.nunique() > 1).sum()))
print("Tamaño del grupo más grande:", inv.groupby("grupo").size().max())

In [ ]:
# Ejemplos de grupos de copias (las de mayor tamaño)
top = inv.groupby("grupo").size().sort_values(ascending=False).index[:3]
fig, axes = plt.subplots(3, 6, figsize=(12, 6.5))
for i, g in enumerate(top):
    sub = inv[inv.grupo == g].head(6)
    for j in range(6):
        axes[i, j].axis("off")
        if j < len(sub):
            r = sub.iloc[j]
            axes[i, j].imshow(Image.open(ROOT / r.path).convert("L"), cmap="gray")
            axes[i, j].set_title(f"{r.clase} {r.w}x{r.h}", fontsize=8)
plt.suptitle("Ejemplos de grupos de copias del mismo corte"); plt.tight_layout()
plt.savefig(FIG / "duplicados_ejemplos.png", dpi=150); plt.show()

## 3. Partición train / validación / test por grupos

Se usa `StratifiedGroupKFold`: todas las imágenes de un mismo grupo de copias quedan en el mismo conjunto y se conserva aproximadamente la proporción de clases. Proporciones aproximadas: 71 % entrenamiento, 14 % validación, 14 % test.

- La **validación** se usa para early stopping y para elegir configuraciones.
- El **test** solo se usa para reportar resultados finales.

Para el experimento de fuga de datos (sección 10) también se genera una partición **aleatoria estratificada** con las mismas proporciones, sin respetar los grupos.

Las particiones se guardan en `data/splits/` para que todas las ejecuciones usen exactamente los mismos conjuntos.

In [ ]:
SPLIT_G = SPLITS / "split_grupos.csv"
SPLIT_R = SPLITS / "split_aleatorio.csv"

if SPLIT_G.exists() and SPLIT_R.exists():
    split_g = pd.read_csv(SPLIT_G); split_r = pd.read_csv(SPLIT_R)
    print("Particiones cargadas desde data/splits/")
else:
    # Partición por grupos
    sgkf = StratifiedGroupKFold(n_splits=CFG["n_folds_test"], shuffle=True, random_state=CFG["seed"])
    rest_idx, test_idx = next(sgkf.split(inv, inv.label, inv.grupo))
    rest = inv.iloc[rest_idx]
    sgkf2 = StratifiedGroupKFold(n_splits=CFG["n_folds_val"], shuffle=True, random_state=CFG["seed"])
    tr_i, va_i = next(sgkf2.split(rest, rest.label, rest.grupo))
    split_g = inv[["path", "clase", "label", "grupo"]].copy()
    split_g["split"] = "train"
    split_g.loc[rest.index[va_i], "split"] = "val"
    split_g.loc[inv.index[test_idx], "split"] = "test"

    # Partición aleatoria estratificada con los mismos tamaños
    n_test, n_val = (split_g.split == "test").sum(), (split_g.split == "val").sum()
    rest_r, test_r = train_test_split(inv.index, test_size=n_test, stratify=inv.label, random_state=CFG["seed"])
    tr_r, va_r = train_test_split(rest_r, test_size=n_val, stratify=inv.label[rest_r], random_state=CFG["seed"])
    split_r = inv[["path", "clase", "label", "grupo"]].copy()
    split_r["split"] = "train"; split_r.loc[va_r, "split"] = "val"; split_r.loc[test_r, "split"] = "test"

    split_g.to_csv(SPLIT_G, index=False); split_r.to_csv(SPLIT_R, index=False)
    print("Particiones creadas y guardadas en data/splits/")

def leakage(split_df: pd.DataFrame) -> dict:
    # Fracción de imágenes de val/test cuyo grupo de copias también aparece en train
    g_train = set(split_df.loc[split_df.split == "train", "grupo"])
    return {s: float(split_df.loc[split_df.split == s, "grupo"].isin(g_train).mean()) for s in ["val", "test"]}

tabla_split = pd.crosstab(split_g.clase, split_g.split).loc[CLASSES, ["train", "val", "test"]]
tabla_split.loc["Total"] = tabla_split.sum()
display(tabla_split)
tabla_split.to_csv(TAB / "particion_por_clase.csv")
print("Fuga en split por grupos   :", {k: f"{v:.1%}" for k, v in leakage(split_g).items()})
print("Fuga en split aleatorio    :", {k: f"{v:.1%}" for k, v in leakage(split_r).items()})
assert leakage(split_g)["test"] == 0.0, "El split por grupos no debería tener fuga"

if QUICK:  # subconjunto pequeño, solo para la prueba de humo
    def _sub(df):
        parts = [g.sample(min(len(g), QUICK_N[s] // 4), random_state=0)
                 for s in ["train", "val", "test"] for _, g in df[df.split == s].groupby("label")]
        return pd.concat(parts)
    split_g, split_r = _sub(split_g), _sub(split_r)
    print("MODO RÁPIDO: subconjunto de", len(split_g), "imágenes")

## 4. Preprocesamiento, aumento de datos y carga

**Preprocesamiento:** las imágenes se convierten a escala de grises (1 canal; el 98 % tiene R=G=B) y se redimensionan a `img_size` x `img_size`. Para acelerar el entrenamiento y evitar problemas de multiproceso en Windows, todas las imágenes preprocesadas se guardan una sola vez en memoria y en `data/cache/` (no versionado).

**Normalización:** media y desviación estándar calculadas solo sobre el conjunto de entrenamiento.

**Aumento de datos (solo entrenamiento):** volteo horizontal, rotación de hasta 10 grados, traslación y escala leves, y variación de brillo y contraste. La variación de brillo y contraste busca que la red no dependa del nivel de intensidad, que difiere entre la clase *healthy* y las clases tumorales (sección 1).

In [ ]:
CACHE_FILE = CACHE / f"imgs_{CFG['img_size']}.npz"

def build_cache(paths, size):
    arr = np.zeros((len(paths), size, size), dtype=np.uint8)
    for i, p in enumerate(tqdm(paths, desc="preprocesando")):
        arr[i] = np.asarray(Image.open(ROOT / p).convert("L").resize((size, size), Image.BILINEAR))
    return arr

all_paths = inv.path.tolist()
if CACHE_FILE.exists():
    z = np.load(CACHE_FILE, allow_pickle=True)
    if list(z["paths"]) == all_paths:
        IMGS = z["imgs"]
    else:
        IMGS = build_cache(all_paths, CFG["img_size"])
else:
    IMGS = build_cache(all_paths, CFG["img_size"])
    np.savez(CACHE_FILE, imgs=IMGS, paths=np.array(all_paths, dtype=object))
PATH2ROW = {p: i for i, p in enumerate(all_paths)}
print("Imágenes en memoria:", IMGS.shape, f"({IMGS.nbytes / 1e6:.0f} MB)")

tr_rows = [PATH2ROW[p] for p in split_g.loc[split_g.split == "train", "path"]]
MEAN = float(IMGS[tr_rows].mean() / 255.0); STD = float(IMGS[tr_rows].std() / 255.0)
print(f"Normalización (train): media={MEAN:.4f}, std={STD:.4f}")

In [ ]:
class MRIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.rows = [PATH2ROW[p] for p in df.path]
        self.labels = torch.tensor(df.label.values, dtype=torch.long)
        self.transform = transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        x = torch.from_numpy(IMGS[self.rows[i]]).unsqueeze(0)  # (1, H, W) uint8
        return self.transform(x), self.labels[i]

base_tf = [T.ToDtype(torch.float32, scale=True)]
norm_tf = [T.Normalize([MEAN], [STD])]
TF_TRAIN = T.Compose(base_tf + [
    T.RandomHorizontalFlip(),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
] + norm_tf)
TF_EVAL = T.Compose(base_tf + norm_tf)

def make_loaders(split_df: pd.DataFrame, augment: bool = True, seed: int = 42):
    g = torch.Generator().manual_seed(seed)
    ds = {s: MRIDataset(split_df[split_df.split == s], TF_TRAIN if (s == "train" and augment) else TF_EVAL)
          for s in ["train", "val", "test"]}
    return {s: DataLoader(ds[s], batch_size=CFG["batch_size"], shuffle=(s == "train"),
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"), generator=g if s == "train" else None)
            for s in ds}

# Visualización de un lote aumentado
xb, yb = next(iter(make_loaders(split_g)["train"]))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, x, y in zip(axes.ravel(), xb, yb):
    ax.imshow((x[0] * STD + MEAN).clamp(0, 1), cmap="gray"); ax.set_title(CLASSES[y], fontsize=8); ax.axis("off")
plt.suptitle("Lote de entrenamiento con aumento de datos"); plt.tight_layout()
plt.savefig(FIG / "lote_aumentado.png", dpi=120); plt.show()

## 5. Modelos

### 5.1 CNN propia
Arquitectura tipo VGG configurable. Cada bloque tiene dos convoluciones `k x k` con Batch Normalization y ReLU, seguidas de max pooling 2x2. El número de filtros se duplica en cada bloque (32, 64, 128, 256). Al final: global average pooling, dropout de 0,5 y una capa lineal de 4 salidas.

Hiperparámetros de arquitectura explorados en la ablación: **número de bloques** (2, 3, 4) y **tamaño de filtro** (3x3, 5x5).

Prácticas tomadas de AlexNet: activación ReLU, max pooling, dropout y aumento de datos. Batch Normalization reemplaza la normalización local de respuesta (LRN) usada en AlexNet.

### 5.2 ResNet18 preentrenada (ImageNet)
Se reemplaza la capa final por dropout + lineal de 4 salidas. La imagen de 1 canal se replica en 3 canales. Dos variantes:
- **Extractor de características:** se congela toda la red salvo la capa final.
- **Fine-tuning completo:** se entrenan todas las capas.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, n_blocks: int = 3, kernel: int = 3, base: int = 32, n_classes: int = 4, dropout: float = 0.5):
        super().__init__()
        layers, c_in = [], 1
        for i in range(n_blocks):
            c_out = min(base * 2 ** i, 256)
            layers.append(nn.Sequential(
                nn.Conv2d(c_in, c_out, kernel, padding=kernel // 2, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(),
                nn.Conv2d(c_out, c_out, kernel, padding=kernel // 2, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(),
                nn.MaxPool2d(2)))
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, n_classes))
    def forward(self, x):
        return self.head(self.features(x))
    def cam_layer(self):
        return self.features[-1][-2]  # última ReLU antes del pooling

class ResNet18TL(nn.Module):
    def __init__(self, freeze_backbone: bool, n_classes: int = 4, dropout: float = 0.3):
        super().__init__()
        try:
            self.net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        except Exception as e:
            if not QUICK:
                raise
            print("MODO RÁPIDO: no se pudieron descargar los pesos preentrenados, se usa inicialización aleatoria.", e)
            self.net = torchvision.models.resnet18(weights=None)
        if freeze_backbone:
            for p in self.net.parameters():
                p.requires_grad = False
        self.net.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.net.fc.in_features, n_classes))
    def forward(self, x):
        return self.net(x.repeat(1, 3, 1, 1))
    def cam_layer(self):
        return self.net.layer4

def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)

arq = []
for nb in [2, 3, 4]:
    for k in [3, 5]:
        tot, _ = count_params(CustomCNN(nb, k))
        arq.append(dict(modelo=f"CNN b{nb} k{k}", parametros=tot))
display(pd.DataFrame(arq))

## 6. Funciones de entrenamiento y evaluación

- **Pérdida:** entropía cruzada. **Optimizador:** AdamW con *cosine annealing* de la tasa de aprendizaje.
- **Early stopping:** se conserva la época con mejor **F1 macro en validación** y se detiene si no mejora en `patience` épocas.
- **Precisión mixta** (AMP) cuando hay GPU.

**Métricas reportadas:**
- *Accuracy* y **F1 macro** (promedio no ponderado por clase; no se ve dominado por la clase más frecuente).
- *Precision*, *recall* y F1 por clase, y matriz de confusión.
- **Tasa de falsos negativos tumor → healthy**: fracción de imágenes con tumor clasificadas como tejido sano. Es el error más grave en un contexto clínico, porque implica no detectar un tumor.

In [ ]:
def compute_metrics(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(len(CLASSES)), zero_division=0)
    tumor = np.isin(y_true, TUMOR_IDX)
    m = dict(accuracy=accuracy_score(y_true, y_pred),
             f1_macro=f1_score(y_true, y_pred, average="macro", labels=range(len(CLASSES)), zero_division=0),
             fn_tumor_a_healthy=float((y_pred[tumor] == HEALTHY_IDX).mean()) if tumor.any() else np.nan)
    for i, c in enumerate(CLASSES):
        m[f"recall_{c}"] = r[i]; m[f"precision_{c}"] = p[i]
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval(); ys, ps, probs, loss_sum, n = [], [], [], 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(x)
        loss_sum += F.cross_entropy(out.float(), y, reduction="sum").item(); n += len(y)
        pr = out.float().softmax(1)
        ys.append(y.cpu()); ps.append(pr.argmax(1).cpu()); probs.append(pr.cpu())
    return torch.cat(ys).numpy(), torch.cat(ps).numpy(), torch.cat(probs).numpy(), loss_sum / max(n, 1)

def fit(model, loaders, epochs, lr, tag):
    ckpt, hist_path = MODELS / f"{tag}.pt", TAB / f"historia_{tag}.csv"
    if CFG["resume"] and ckpt.exists() and hist_path.exists():
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"[{tag}] pesos cargados desde {ckpt.name} (no se reentrena)")
        return model.to(DEVICE), pd.read_csv(hist_path)

    model = model.to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    best_f1, best_state, wait, hist = -1, None, 0, []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); tl, tc, tn = 0.0, 0, 0
        for x, y in loaders["train"]:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
                loss = F.cross_entropy(out, y, label_smoothing=CFG["label_smoothing"])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        sched.step()
        yv, pv, _, vl = predict(model, loaders["val"])
        mv = compute_metrics(yv, pv)
        hist.append(dict(epoch=ep, train_loss=tl / tn, train_acc=tc / tn, val_loss=vl,
                         val_acc=mv["accuracy"], val_f1=mv["f1_macro"], lr=sched.get_last_lr()[0]))
        print(f"[{tag}] ep {ep:02d} | train loss {tl/tn:.3f} acc {tc/tn:.3f} | val loss {vl:.3f} "
              f"acc {mv['accuracy']:.3f} F1 {mv['f1_macro']:.3f}")
        if mv["f1_macro"] > best_f1:
            best_f1, best_state, wait = mv["f1_macro"], copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= CFG["patience"]:
                print(f"[{tag}] early stopping en la época {ep}"); break
    hist = pd.DataFrame(hist); hist["tiempo_s"] = time.time() - t0
    model.load_state_dict(best_state)
    torch.save(best_state, ckpt); hist.to_csv(hist_path, index=False)
    return model, hist

RESULTS, PREDS = [], {}
RES_PATH = TAB / "resultados_todos.csv"

def run(tag, model_fn, split_df, epochs, lr, seed, extra=None):
    set_seed(seed)
    loaders = make_loaders(split_df, seed=seed)
    model = model_fn()
    model, hist = fit(model, loaders, epochs, lr, tag)
    best = hist.loc[hist.val_f1.idxmax()]
    row = dict(tag=tag, seed=seed, epocas=len(hist), mejor_epoca=int(best.epoch),
               train_acc=best.train_acc, val_acc=best.val_acc, val_f1=best.val_f1,
               brecha_train_val=best.train_acc - best.val_acc,
               parametros_entrenables=count_params(model)[1], tiempo_s=float(hist.tiempo_s.iloc[0]))
    yt, pt, prt, _ = predict(model, loaders["test"])
    row.update({f"test_{k}": v for k, v in compute_metrics(yt, pt).items()})
    row.update(extra or {})
    PREDS[tag] = dict(y=yt, pred=pt, prob=prt, paths=split_df.loc[split_df.split == "test", "path"].values)
    RESULTS[:] = [r for r in RESULTS if r["tag"] != tag] + [row]
    pd.DataFrame(RESULTS).to_csv(RES_PATH, index=False)
    return model, hist, row

## 7. Experimento 1: ablación de la CNN propia

Se entrena la CNN propia con todas las combinaciones de **número de bloques** y **tamaño de filtro**, con la misma partición, semilla y presupuesto de épocas. La mejor configuración se elige por **F1 macro en validación** (nunca con test).

La **brecha train - validación** (accuracy de entrenamiento menos accuracy de validación en la mejor época) se usa como indicador de sobreajuste.

In [ ]:
HIST = {}
for nb in CFG["ablation_blocks"]:
    for k in CFG["ablation_kernels"]:
        tag = f"cnn_b{nb}_k{k}_s{CFG['seed']}"
        _, HIST[tag], _ = run(tag, lambda nb=nb, k=k: CustomCNN(nb, k), split_g, CFG["epochs_cnn"], CFG["lr_cnn"],
                              CFG["seed"], extra=dict(modelo="CNN propia", bloques=nb, kernel=k, split="grupos"))

abl = pd.DataFrame([r for r in RESULTS if r.get("modelo") == "CNN propia" and r["split"] == "grupos" and r["seed"] == CFG["seed"]])
cols = ["bloques", "kernel", "parametros_entrenables", "mejor_epoca", "train_acc", "val_acc", "val_f1", "brecha_train_val", "tiempo_s"]
abl = abl[cols].sort_values(["bloques", "kernel"]).reset_index(drop=True)
abl.to_csv(TAB / "exp1_ablacion_cnn.csv", index=False)
display(abl)

best_cfg = abl.loc[abl.val_f1.idxmax()]
BEST_NB, BEST_K = int(best_cfg.bloques), int(best_cfg.kernel)
print(f"Mejor configuración (val F1): {BEST_NB} bloques, kernel {BEST_K}x{BEST_K} -> val F1 = {best_cfg.val_f1:.4f}")

In [ ]:
# Curvas de entrenamiento de la ablación y mapa de F1 en validación
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for tag, h in HIST.items():
    name = tag.replace(f"_s{CFG['seed']}", "").replace("cnn_", "")
    color = ax[0].plot(h.epoch, h.val_loss, marker=".", label=name)[0].get_color()
    ax[0].plot(h.epoch, h.train_loss, "--", alpha=0.6, color=color)
    ax[1].plot(h.epoch, h.val_f1, marker=".", color=color, label=name)
ax[0].set_title("Pérdida (línea punteada: train)"); ax[0].set_xlabel("Época"); ax[0].legend(fontsize=7)
ax[1].set_title("F1 macro en validación"); ax[1].set_xlabel("Época"); ax[1].legend(fontsize=7)
piv = abl.pivot(index="bloques", columns="kernel", values="val_f1")
im = ax[2].imshow(piv.values, cmap="viridis")
ax[2].set_xticks(range(len(piv.columns)), [f"{k}x{k}" for k in piv.columns]); ax[2].set_yticks(range(len(piv.index)), piv.index)
ax[2].set_xlabel("Tamaño de filtro"); ax[2].set_ylabel("Bloques"); ax[2].set_title("F1 macro en validación")
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax[2].text(j, i, f"{piv.values[i, j]:.3f}", ha="center", va="center", color="w")
plt.colorbar(im, ax=ax[2]); plt.tight_layout(); plt.savefig(FIG / "exp1_ablacion.png", dpi=200); plt.show()

## 8. Experimento 2: transferencia de aprendizaje con ResNet18

Se comparan dos estrategias con pesos de ImageNet: congelar la red y entrenar solo la capa final, o hacer fine-tuning de todas las capas. Se usa una tasa de aprendizaje menor (`lr_tl`) para no destruir las características preentrenadas.

In [ ]:
for freeze in [True, False]:
    tag = f"resnet18_{'congelada' if freeze else 'finetune'}_s{CFG['seed']}"
    lr = CFG["lr_tl"] * (10 if freeze else 1)  # la capa final sola tolera una tasa mayor
    _, HIST[tag], _ = run(tag, lambda freeze=freeze: ResNet18TL(freeze), split_g, CFG["epochs_tl"], lr, CFG["seed"],
                          extra=dict(modelo="ResNet18 " + ("congelada" if freeze else "fine-tuning"), split="grupos"))

tl = pd.DataFrame([r for r in RESULTS if str(r.get("modelo", "")).startswith("ResNet18") and r["seed"] == CFG["seed"]])
display(tl[["modelo", "parametros_entrenables", "mejor_epoca", "train_acc", "val_acc", "val_f1", "brecha_train_val", "tiempo_s"]])
tl.to_csv(TAB / "exp2_transfer_learning.csv", index=False)
BEST_TL = tl.loc[tl.val_f1.idxmax(), "modelo"]
BEST_TL_FREEZE = BEST_TL.endswith("congelada")
print("Mejor variante de transferencia (val F1):", BEST_TL)

## 9. Experimento 3: robustez con varias semillas

La mejor CNN propia y la mejor variante de ResNet18 se reentrenan con distintas semillas (inicialización, orden de los lotes y aumento de datos) sobre la misma partición. Se reporta **media ± desviación estándar** en test.

In [ ]:
for seed in CFG["seeds"]:
    run(f"cnn_b{BEST_NB}_k{BEST_K}_s{seed}", lambda: CustomCNN(BEST_NB, BEST_K), split_g, CFG["epochs_cnn"], CFG["lr_cnn"], seed,
        extra=dict(modelo="CNN propia", bloques=BEST_NB, kernel=BEST_K, split="grupos"))
    lr = CFG["lr_tl"] * (10 if BEST_TL_FREEZE else 1)
    run(f"resnet18_{'congelada' if BEST_TL_FREEZE else 'finetune'}_s{seed}", lambda: ResNet18TL(BEST_TL_FREEZE), split_g,
        CFG["epochs_tl"], lr, seed, extra=dict(modelo=BEST_TL, split="grupos"))

R = pd.DataFrame(RESULTS)
sel = R[(R.split == "grupos") & (R.seed.isin(CFG["seeds"])) &
        (((R.modelo == "CNN propia") & (R.bloques == BEST_NB) & (R.kernel == BEST_K)) | (R.modelo == BEST_TL))]
sel = sel.assign(modelo=np.where(sel.modelo == "CNN propia", f"CNN propia ({BEST_NB} bloques, {BEST_K}x{BEST_K})", sel.modelo))
metric_cols = ["test_accuracy", "test_f1_macro", "test_fn_tumor_a_healthy"] + [f"test_recall_{c}" for c in CLASSES]
agg = sel.groupby("modelo")[metric_cols].agg(["mean", "std"])
fmt = pd.DataFrame({c: agg[c]["mean"].map("{:.4f}".format) + " ± " + agg[c]["std"].map("{:.4f}".format) for c in metric_cols})
fmt.columns = [c.replace("test_", "") for c in metric_cols]
fmt["n_semillas"] = sel.groupby("modelo").size()
display(fmt)
fmt.to_csv(TAB / "exp3_semillas_media_std.csv")

## 10. Experimento 4: efecto de la fuga de datos

Se entrena la mejor CNN propia (misma semilla y configuración) sobre la **partición aleatoria**, que no respeta los grupos de copias, y se compara con el resultado de la **partición por grupos**. Además, dentro del test aleatorio se compara el desempeño en imágenes que tienen una copia en entrenamiento frente a las que no.

Si el desempeño con la partición aleatoria es mayor, la diferencia estima cuánto se sobrestima la capacidad de generalización cuando no se controlan los duplicados.

In [ ]:
tag_r = f"cnn_b{BEST_NB}_k{BEST_K}_s{CFG['seed']}_splitaleatorio"
run(tag_r, lambda: CustomCNN(BEST_NB, BEST_K), split_r, CFG["epochs_cnn"], CFG["lr_cnn"], CFG["seed"],
    extra=dict(modelo="CNN propia", bloques=BEST_NB, kernel=BEST_K, split="aleatorio"))

tag_g = f"cnn_b{BEST_NB}_k{BEST_K}_s{CFG['seed']}"
R = pd.DataFrame(RESULTS).set_index("tag")
fuga = pd.DataFrame({
    "Partición por grupos": R.loc[tag_g, ["test_accuracy", "test_f1_macro", "test_recall_healthy", "test_fn_tumor_a_healthy"]],
    "Partición aleatoria":  R.loc[tag_r, ["test_accuracy", "test_f1_macro", "test_recall_healthy", "test_fn_tumor_a_healthy"]],
}).T
fuga.insert(0, "% test con copia en train", [leakage(split_g)["test"], leakage(split_r)["test"]])
display(fuga)
fuga.to_csv(TAB / "exp4_fuga_de_datos.csv")

# Desempeño dentro del test aleatorio: imágenes con copia en train vs sin copia
p = PREDS[tag_r]
g_train = set(split_r.loc[split_r.split == "train", "grupo"])
test_r = split_r[split_r.split == "test"].set_index("path").loc[p["paths"]]
has_copy = test_r.grupo.isin(g_train).values
detalle = pd.DataFrame({
    "n": [has_copy.sum(), (~has_copy).sum()],
    "accuracy": [accuracy_score(p["y"][has_copy], p["pred"][has_copy]) if has_copy.any() else np.nan,
                 accuracy_score(p["y"][~has_copy], p["pred"][~has_copy])],
}, index=["Con copia en train", "Sin copia en train"])
display(detalle)
detalle.to_csv(TAB / "exp4_fuga_detalle.csv")

## 11. Resultados cuantitativos finales

Tabla resumen de todos los experimentos y matrices de confusión en test de los mejores modelos (semilla principal).

In [ ]:
R = pd.DataFrame(RESULTS)
cols = ["tag", "modelo", "split", "seed", "parametros_entrenables", "val_f1", "test_accuracy", "test_f1_macro",
        "test_fn_tumor_a_healthy", "brecha_train_val", "tiempo_s"]
display(R[cols].sort_values(["split", "modelo", "tag"]).reset_index(drop=True))
R.to_csv(TAB / "resultados_todos.csv", index=False)

BEST_TAGS = {f"CNN propia ({BEST_NB} bloques, {BEST_K}x{BEST_K})": f"cnn_b{BEST_NB}_k{BEST_K}_s{CFG['seed']}",
             BEST_TL: f"resnet18_{'congelada' if BEST_TL_FREEZE else 'finetune'}_s{CFG['seed']}"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (name, tag) in zip(axes, BEST_TAGS.items()):
    p = PREDS[tag]
    cm = confusion_matrix(p["y"], p["pred"], labels=range(len(CLASSES)))
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f"{cm[i, j]}\n({cmn[i, j]:.0%})", ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=30); ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real")
    ax.set_title(f"{name}\nacc={accuracy_score(p['y'], p['pred']):.3f}, F1 macro={f1_score(p['y'], p['pred'], average='macro'):.3f}", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "matrices_confusion.png", dpi=200); plt.show()

for name, tag in BEST_TAGS.items():
    p = PREDS[tag]
    print(f"=== {name} (test) ===")
    print(classification_report(p["y"], p["pred"], labels=range(len(CLASSES)), target_names=CLASSES, digits=4, zero_division=0))
    pd.DataFrame(classification_report(p["y"], p["pred"], labels=range(len(CLASSES)), target_names=CLASSES,
                                       output_dict=True, zero_division=0)).T.to_csv(TAB / f"reporte_clasificacion_{tag}.csv")

## 12. Resultados cualitativos y Grad-CAM

Para el mejor modelo según F1 macro en test (semilla principal) se muestran, por clase, ejemplos clasificados correctamente y errores, con la clase predicha y la confianza.

**Grad-CAM** resalta las regiones de la imagen que más influyen en la predicción. Permite verificar si la red se fija en la lesión o en elementos ajenos a ella (bordes, texto, fondo).

In [ ]:
best_name = max(BEST_TAGS, key=lambda n: f1_score(PREDS[BEST_TAGS[n]]["y"], PREDS[BEST_TAGS[n]]["pred"], average="macro"))
best_tag = BEST_TAGS[best_name]
p = PREDS[best_tag]
print("Modelo para análisis cualitativo:", best_name)

def show_img(ax, path, title, color="black"):
    ax.imshow(IMGS[PATH2ROW[path]], cmap="gray"); ax.set_title(title, fontsize=7, color=color); ax.axis("off")

rng = np.random.default_rng(CFG["seed"])
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(12, 8.5))
for i, c in enumerate(CLASSES):
    ok = np.where((p["y"] == i) & (p["pred"] == i))[0]
    bad = np.where((p["y"] == i) & (p["pred"] != i))[0]
    pick = list(rng.choice(ok, min(3, len(ok)), replace=False)) + list(rng.choice(bad, min(3, len(bad)), replace=False))
    for j in range(6):
        axes[i, j].axis("off")
        if j < len(pick):
            k = pick[j]; correct = p["pred"][k] == p["y"][k]
            show_img(axes[i, j], p["paths"][k], f"real: {c}\npred: {CLASSES[p['pred'][k]]} ({p['prob'][k].max():.2f})",
                     "green" if correct else "red")
    axes[i, 0].text(-0.15, 0.5, c, transform=axes[i, 0].transAxes, rotation=90, va="center", fontsize=10)
plt.suptitle(f"{best_name}: aciertos (verde) y errores (rojo) en test"); plt.tight_layout()
plt.savefig(FIG / "cualitativo_aciertos_errores.png", dpi=200); plt.show()

In [ ]:
def grad_cam(model, x, layer):
    acts, grads = {}, {}
    h1 = layer.register_forward_hook(lambda m, i, o: acts.__setitem__("a", o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__("g", go[0]))
    model.eval(); x = x.clone().requires_grad_(True)
    out = model(x); idx = out.argmax(1)
    model.zero_grad(); out[torch.arange(len(x)), idx].sum().backward()
    h1.remove(); h2.remove()
    w = grads["g"].mean((2, 3), keepdim=True)
    cam = F.relu((w * acts["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    cam = cam / (cam.amax((1, 2), keepdim=True) + 1e-8)
    return cam.detach().cpu().numpy(), idx.cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for col, (name, tag) in enumerate(BEST_TAGS.items()):
    is_cnn = tag.startswith("cnn")
    model = CustomCNN(BEST_NB, BEST_K) if is_cnn else ResNet18TL(BEST_TL_FREEZE)
    model.load_state_dict(torch.load(MODELS / f"{tag}.pt", map_location=DEVICE)); model.to(DEVICE)
    pp = PREDS[tag]
    idxs = [np.where(pp["y"] == i)[0][0] for i in range(len(CLASSES)) if (pp["y"] == i).any()]
    x = torch.stack([TF_EVAL(torch.from_numpy(IMGS[PATH2ROW[pp["paths"][k]]]).unsqueeze(0)) for k in idxs]).to(DEVICE)
    cams, pred = grad_cam(model, x, model.cam_layer())
    for j, k in enumerate(idxs):
        ax_img, ax_cam = axes[col, 2 * j], axes[col, 2 * j + 1]
        show_img(ax_img, pp["paths"][k], f"real: {CLASSES[pp['y'][k]]}")
        ax_cam.imshow(IMGS[PATH2ROW[pp["paths"][k]]], cmap="gray"); ax_cam.imshow(cams[j], cmap="jet", alpha=0.45)
        ax_cam.set_title(f"pred: {CLASSES[pred[j]]}", fontsize=7); ax_cam.axis("off")
    axes[col, 0].text(-0.2, 0.5, name, transform=axes[col, 0].transAxes, rotation=90, va="center", fontsize=8)
plt.suptitle("Grad-CAM: una imagen de test por clase"); plt.tight_layout()
plt.savefig(FIG / "gradcam.png", dpi=200); plt.show()

## 13. Resumen de cifras para el informe

Las cifras de esta celda son las que deben aparecer en el informe, para que el notebook y el informe coincidan.

In [ ]:
R = pd.DataFrame(RESULTS).set_index("tag")
resumen_final = {
    "dataset": {"imagenes": int(len(inv)), "cortes_unicos": int(inv.grupo.nunique()),
                "redundancia_healthy_%": float(dup.loc["healthy", "%_redundante"])},
    "particion_por_grupos": tabla_split.loc["Total"].to_dict(),
    "mejor_cnn": {"bloques": BEST_NB, "kernel": BEST_K},
    "mejor_transfer": BEST_TL,
    "test_semilla_principal": {name: {k: round(float(R.loc[tag, f"test_{k}"]), 4)
                                      for k in ["accuracy", "f1_macro", "fn_tumor_a_healthy"]}
                               for name, tag in BEST_TAGS.items()},
    "fuga_de_datos": fuga[["test_accuracy", "test_f1_macro"]].round(4).to_dict(orient="index"),
    "dispositivo": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
    "modo_rapido": QUICK,
}
(TAB / "resumen_informe.json").write_text(json.dumps(resumen_final, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(json.dumps(resumen_final, indent=2, ensure_ascii=False, default=str))
print("\nArchivos generados:")
for f in sorted(list(FIG.glob("*.png")) + list(TAB.glob("*.csv")) + list(TAB.glob("*.json"))):
    print(" ", f.relative_to(ROOT))